<a href="https://colab.research.google.com/github/Jean1489/CLAIR/blob/main/notebooks/02_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!git clone https://github.com/Jean1489/CLAIR.git

Cloning into 'CLAIR'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 27 (delta 13), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 555.65 KiB | 20.58 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [4]:
%cd /content/CLAIR
!git config user.email "yhan.trujillo@correounivalle.edu.co"
!git config user.name "Jean1489"

/content/CLAIR


In [5]:
import os
import torch
import torchvision
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

In [6]:
# Paths
BASE_PATH: str = '/content/drive/MyDrive/CLAIR/data/raw'
TRAIN_PATH: str = f'{BASE_PATH}/Training'
TEST_PATH: str = f'{BASE_PATH}/Testing'

# Dataset
CLASS_NAMES: list[str] = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
NUM_CLASSES: int = 4
IMG_SIZE: int = 224

# Training
BATCH_SIZE: int = 32
NUM_EPOCHS: int = 30
LEARNING_RATE: float = 1e-4

# Device
DEVICE: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


In [7]:
class BrainTumorDataset(Dataset):

    def __init__(self, root_path: str, transform=None) -> None:
        self.root_path = root_path
        self.transform = transform
        self.samples: list[tuple[str, int]] = []

        for label, class_name in enumerate(CLASS_NAMES):
            class_path = os.path.join(root_path, class_name)
            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)
                self.samples.append((img_path, label))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [8]:
# ImageNet normalization values
IMAGENET_MEAN: list[float] = [0.485, 0.456, 0.406]
IMAGENET_STD: list[float]  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [9]:
from torch.utils.data import random_split

# Create full training dataset
full_dataset = BrainTumorDataset(
    root_path=TRAIN_PATH,
    transform=train_transforms
)

# Calculate split sizes
total: int = len(full_dataset)
train_size: int = int(0.70 * total)
val_size: int = total - train_size

# Split dataset
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Test dataset
test_dataset = BrainTumorDataset(
    root_path=TEST_PATH,
    transform=val_transforms
)

print(f'Train: {len(train_dataset)} images')
print(f'Val:   {len(val_dataset)} images')
print(f'Test:  {len(test_dataset)} images')

Train: 2008 images
Val:   862 images
Test:  394 images


In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print('DataLoaders created successfully')

DataLoaders created successfully


In [11]:
# Get one batch and verify shapes
images, labels = next(iter(train_loader))

print(f'Batch shape: {images.shape}')
print(f'Labels shape: {labels.shape}')
print(f'Labels: {labels}')

Batch shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Labels: tensor([1, 2, 3, 1, 1, 1, 0, 3, 3, 1, 0, 0, 3, 0, 1, 0, 1, 3, 2, 3, 3, 2, 2, 3,
        2, 2, 0, 3, 0, 2, 3, 3])
